# Soup Kernel Challenge — Мерей: pinned working copy

This notebook clones `https://github.com/promaladoi/Decentrathon-6.0` at code commit `66bcd9b6585b027561a158448d87c2699f975931` and
runs from its `Мерей` folder. Enable **GPU T4 x2** and **Internet**; keep
both GPUs idle before Run All. It verifies the exact measured source hash
before tests and executes the unchanged official harness (base, solution,
solution, base). The measured development result is [saved version
363884710](https://www.kaggle.com/code/promaladoi/soup-kaggle-development?scriptVersionId=363884710),
which used the immutable embedded source snapshot. **This clone notebook
is prepared for reproduction and has not itself been executed.**

The original platform template stays in `templates/kaggle_notebook.ipynb`.
Final submission still needs a platform-repository commit and a saved public
notebook pinned to that commit. Optional `GITHUB_TOKEN` and `HF_TOKEN`
come from Kaggle Secrets; no credentials are stored in this notebook.


In [ ]:
REPO_URL = "https://github.com/promaladoi/Decentrathon-6.0.git"
COMMIT = "66bcd9b6585b027561a158448d87c2699f975931"
EXPECTED_SOLUTION_SHA256 = "60216a16a1f993117f90e6028e93e4096fbacc4fb21afefbccd3f7f3a2e3a7f0"
CHECKOUT = "/kaggle/working/merey-pinned-66bcd9b6585b"


In [ ]:
import os, subprocess, sys
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True, check=True).stdout
print(gpu)
assert "T4" in gpu, "Choose GPU T4 x2"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
               "soup-cli[train]==0.77.0", "transformers==5.16.1", "peft==0.20.0",
               "trl==0.29.1", "bitsandbytes==0.50.2", "accelerate==1.14.0", "pytest"], check=True)


In [ ]:
import base64
from pathlib import Path
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
auth = []
if token:
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth = ["-c", f"http.extraHeader=Authorization: Basic {basic}"]
if not Path(CHECKOUT).exists():
    cloned = subprocess.run(["git", *auth, "clone", "-q", REPO_URL, CHECKOUT],
                            capture_output=True, text=True)
    if cloned.returncode:
        raise RuntimeError("Git clone failed; check repository access and the optional Kaggle secret")
subprocess.run(["git", "-C", CHECKOUT, "checkout", "-q", COMMIT], check=True)
actual_commit = subprocess.run(["git", "-C", CHECKOUT, "rev-parse", "HEAD"],
                              capture_output=True, text=True, check=True).stdout.strip()
assert actual_commit == COMMIT
os.chdir(Path(CHECKOUT) / "Мерей")
print("Commit:", actual_commit, "Working folder:", Path.cwd())
del token, auth


In [ ]:
import hashlib
import torch, triton
assert torch.__version__ == '2.11.0+cu128', "Use the recorded Kaggle image/PyTorch version"
assert triton.__version__ == '3.6.0', "Use the recorded Triton version"
digest = hashlib.sha256()
for path in sorted(Path("solution").rglob("*")):
    if path.is_file() and not any(p in ("__pycache__", ".pytest_cache") for p in path.parts):
        digest.update(path.relative_to("solution").as_posix().encode())
        digest.update(path.read_bytes())
assert digest.hexdigest() == EXPECTED_SOLUTION_SHA256, "Clone differs from measured solution bytes"
print("Measured solution SHA256 verified:", digest.hexdigest())
sys.path.insert(0, str(Path("harness").resolve()))
from common import ENV
def run_repo(arguments, **overrides):
    environment = dict(ENV)
    environment.update(overrides)
    subprocess.run([sys.executable, *arguments], env=environment, check=True)
run_repo(["task/prepare_data.py"])
print("SOLUTION_UNIT_TESTS_BEGIN", flush=True)
run_repo(["-m", "pytest", "solution", "-q"])
print("SOUP_REGRESSION_TESTS_BEGIN", flush=True)
run_repo(["-m", "pytest", "tests", "-q"], SOUP_APPLY_SOLUTION="1")


In [ ]:
run_repo(["harness/run_task.py", "--repeats", "2"])


In [ ]:
run_repo(["harness/check_rules.py"])
run_repo(["harness/score.py", "runs/result.json"])


Keep runs/result.json, runs/checks.json and the full saved Logs. The same-team A/B comparison in the recorded development notebook is a separate performance diagnostic; the official score uses only native Soup versus this candidate.
